The advertising company "Greedy Oligarchs & Other Grubby Lowdown Evildoers" has $n$ clients and wants to maximise their revenues before their initial public offering in $T$ days. They decide to concentrate each day on a single advertising campaign for a single client. For example, on day 1 they might advertise for client 3, on day 2 for client 7 and day 3 back to client 3 again. Some clients are bigger than others and so make more money, but repeatedly advertising for the same clients has diminishing returns. The first day of advertising for client $i$ they will make $c_i$. The second day of advertising for the same client, $c_i/2$, the third day $c_i/4$ and so on, at day $k$ for client $i$ they will make $c_i/2^k$, so it might make sense for them to advertise one day with one client, and then another day with another client, and so on.

1. Devise a greedy algorithm to maximise their profit, but do not worry about efficiency of your program for the moment. Describe your program in words.

2. Prove that the greedy choice property holds.

3. What is the complexity of your algorithm? Note you should give your answer in terms of $n$ and $T$. (This two variable complexity notation is not examinable.)

4. (optional) Look up the [Fibonacci heap priority queue](https://en.wikipedia.org/wiki/Fibonacci_heap). Use this to improve your algorithm. Write code or pseudocode, and compute the complexity of your algorithm. Find a solution when ``c=[2, 3, 5]`` and ``T=10``.

Question 1: Because we don't care about the performance, we can then use a simple method. We first list the entire client, and sort them using certain algorithm provided by python std. we pick the largest one at that day and update it, and sort again in the second day. we continue this process until we achieve the goal. Therefore, we will have the complexity = O(T)*O(nlogn) = O(Tnlogn)

Question 2: The greedy choice property states that we can make best choice without looking at alternatives. In this case, the day k for client i is counted individually. Hence, no matter who we choice, and we choice again how many days later. we will still have the same results. therefore, the greedy choice property holds here.

**Reference Answer for Q2:**

We prove the greedy choice property using an exchange argument.

**Claim:** At each step, choosing the client with the maximum current revenue is part of some optimal solution.

**Proof:** Suppose there exists an optimal solution OPT that, on some day $d$, does not pick the client with the highest available revenue. Let the greedy algorithm pick client $A$ on day $d$ (with revenue $r_A$), while OPT picks client $B$ (with revenue $r_B < r_A$).

Since the revenue each client generates depends only on how many times that client has been chosen (not on when), the order of selections does not affect the individual revenues earned from each client. Therefore, we can swap the selection of $A$ and $B$ between day $d$ and whatever day OPT would have picked $A$ (or simply replace $B$ with $A$ if OPT never picks $A$ on that occasion).

After the swap, the total revenue does not decrease, because:
- If OPT picked $A$ on some later day $d'$: swapping the assignments of days $d$ and $d'$ does not change the total revenue (since each client's revenue depends only on how many times it is selected, not when).
- If OPT never picked $A$ at all in place of this instance: replacing $B$ with $A$ on day $d$ gives $r_A > r_B$, strictly increasing total revenue, contradicting OPT's optimality.

In both cases, we obtain a solution that is at least as good as OPT and agrees with the greedy choice on day $d$. Repeating this argument for every day shows that the greedy solution is optimal. $\blacksquare$

Question 3: O(T) * O(nlogn)

In [1]:
print(1)

1


In [2]:
import heapq
c = [2, 3, 5]
T = 10

heap = []
for i, value in enumerate(c):
    heapq.heappush(heap, (-value, i))
    
def greedy_1(heap, T):
    while T > 0:
        tmp = 0
        tmp, client = heapq.heappop(heap)
        heapq.heappush(heap, (tmp/2, client))
        print(f"Visit{client}, get {-tmp}")
        T -= 1
        
greedy_1(heap, T)



Visit2, get 5
Visit1, get 3
Visit2, get 2.5
Visit0, get 2
Visit1, get 1.5
Visit2, get 1.25
Visit0, get 1.0
Visit1, get 0.75
Visit2, get 0.625
Visit0, get 0.5


In [ ]:
# Reference Answer for Q4: using heap with complexity analysis
# Complexity: O(n) to build heap + O(T log n) for T extract-max and insert operations
# Total: O(n + T log n), improved from O(T * n log n) in the naive sorting approach.
# With a Fibonacci heap, insert is O(1) amortized, extract-max is O(log n) amortized,
# so overall complexity remains O(n + T log n).

import heapq

c = [2, 3, 5]
T = 10

# Build max-heap (negate values since heapq is a min-heap)
heap = [(-value, i) for i, value in enumerate(c)]
heapq.heapify(heap)  # O(n) heapify instead of n individual pushes

total_revenue = 0
for day in range(1, T + 1):
    neg_revenue, client = heapq.heappop(heap)       # O(log n)
    revenue = -neg_revenue
    total_revenue += revenue
    heapq.heappush(heap, (neg_revenue / 2, client))  # O(log n)
    print(f"Day {day}: advertise client {client}, revenue = {revenue}")

print(f"\nTotal revenue: {total_revenue}")

In [ ]:
# Reference: Fibonacci Heap 实现 + 用于本题
# 
# Fibonacci Heap vs Binary Heap 关键区别:
#   操作          Binary Heap    Fibonacci Heap
#   insert        O(log n)       O(1) amortized
#   find-max      O(1)           O(1)
#   extract-max   O(log n)       O(log n) amortized
#   decrease-key  O(log n)       O(1) amortized  ← Fibonacci heap 的杀手锏
#
# 本题中: 每轮 extract-max O(log n) + insert O(1) = O(log n)
# 总复杂度: O(n + T log n)，和 binary heap 的 O(n + T·2log n) 相比只省了常数因子
# Fibonacci heap 真正发光的场景是 Dijkstra / Prim，那里 decrease-key 被大量调用

import math

class FibNode:
    def __init__(self, key, value=None):
        self.key = key          # 优先级 (越大越优先)
        self.value = value
        self.degree = 0         # 子节点数量
        self.parent = None
        self.child = None       # 指向某个子节点
        self.left = self        # 双向链表
        self.right = self
        self.mark = False       # 是否丢失过子节点

class FibonacciMaxHeap:
    """Max-heap variant of Fibonacci Heap"""
    
    def __init__(self):
        self.max_node = None
        self.n = 0
    
    def _add_to_root_list(self, node):
        """将 node 插入根链表"""
        if self.max_node is None:
            node.left = node
            node.right = node
            self.max_node = node
        else:
            node.right = self.max_node.right
            node.left = self.max_node
            self.max_node.right.left = node
            self.max_node.right = node
            if node.key > self.max_node.key:
                self.max_node = node
    
    def insert(self, key, value=None):
        """O(1) - 直接加入根链表"""
        node = FibNode(key, value)
        node.parent = None
        node.child = None
        node.degree = 0
        node.mark = False
        self._add_to_root_list(node)
        self.n += 1
        return node
    
    def find_max(self):
        """O(1) - 返回最大节点"""
        return self.max_node
    
    def _remove_from_list(self, node):
        """从双向链表中移除节点"""
        node.left.right = node.right
        node.right.left = node.left
    
    def _link(self, child, parent):
        """将 child 变为 parent 的子节点"""
        self._remove_from_list(child)
        child.parent = parent
        if parent.child is None:
            parent.child = child
            child.left = child
            child.right = child
        else:
            child.right = parent.child.right
            child.left = parent.child
            parent.child.right.left = child
            parent.child.right = child
        parent.degree += 1
        child.mark = False
    
    def _consolidate(self):
        """合并根链表中 degree 相同的树 — extract-max 的核心"""
        max_degree = int(math.log2(self.n)) + 2
        A = [None] * max_degree
        
        # 收集所有根节点
        roots = []
        curr = self.max_node
        if curr:
            while True:
                roots.append(curr)
                curr = curr.right
                if curr == self.max_node:
                    break
        
        for w in roots:
            x = w
            d = x.degree
            while d < max_degree and A[d] is not None:
                y = A[d]
                if x.key < y.key:
                    x, y = y, x
                self._link(y, x)
                A[d] = None
                d += 1
            if d < max_degree:
                A[d] = x
        
        self.max_node = None
        for node in A:
            if node is not None:
                node.left = node
                node.right = node
                node.parent = None
                self._add_to_root_list(node)
    
    def extract_max(self):
        """O(log n) amortized - 取出最大值，然后 consolidate"""
        z = self.max_node
        if z is None:
            return None
        
        # 将 z 的所有子节点加入根链表
        if z.child:
            children = []
            curr = z.child
            while True:
                children.append(curr)
                curr = curr.right
                if curr == z.child:
                    break
            for child in children:
                self._remove_from_list(child)
                child.parent = None
                self._add_to_root_list(child)
        
        # 从根链表移除 z
        if z == z.right:
            self.max_node = None
        else:
            self.max_node = z.right
            self._remove_from_list(z)
            self._consolidate()
        
        self.n -= 1
        return z


# ====== 用 Fibonacci Heap 解决本题 ======
c = [2, 3, 5]
T = 10

fib_heap = FibonacciMaxHeap()
for i, value in enumerate(c):
    fib_heap.insert(value, i)        # O(1) each

total_revenue = 0
for day in range(1, T + 1):
    node = fib_heap.extract_max()    # O(log n) amortized
    revenue = node.key
    client = node.value
    total_revenue += revenue
    fib_heap.insert(revenue / 2, client)  # O(1) amortized ← 这里比 binary heap 快!
    print(f"Day {day}: advertise client {client}, revenue = {revenue}")

print(f"\nTotal revenue: {total_revenue}")